In [ ]:
pip install transformers datasets torch scikit-learn gradio


In [ ]:
from datasets import load_dataset
from transformers import BertTokenizer

ds = load_dataset("sh0416/ag_news")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    # Concatenate 'title' and 'description' for each example in the batch
    return tokenizer(
        [t + " " + d for t, d in zip(examples["title"], examples["description"])],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = ds.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["title", "description"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
# Adjust labels to be 0-indexed
def adjust_labels(examples):
    examples["labels"] = [label - 1 for label in examples["labels"]]
    return examples
tokenized_dataset = tokenized_dataset.map(adjust_labels, batched=True)
tokenized_dataset.set_format("torch")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/33.7M [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch", # Changed from evaluation_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
)

trainer.train()


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Epoch,Training Loss,Validation Loss
1,0.194700,0.182504
2,0.134500,0.185131
3,0.077900,0.228200


TrainOutput(global_step=22500, training_loss=0.14994759487575954, metrics={'train_runtime': 9427.1413, 'train_samples_per_second': 38.188, 'train_steps_per_second': 2.387, 'total_flos': 2.368042020864e+16, 'train_loss': 0.14994759487575954, 'epoch': 3.0})

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

predictions = trainer.predict(tokenized_dataset["test"])

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average="weighted")

print("Accuracy:", accuracy)
print("F1 Score:", f1)


Accuracy: 0.9481578947368421
F1 Score: 0.94819183524289


In [ ]:
model.save_pretrained("/content/drive/MyDrive/News topic classifier/saved_model")
tokenizer.save_pretrained("/content/drive/MyDrive/News topic classifier/saved_model")

('/content/drive/MyDrive/News topic classifier/saved_model/tokenizer_config.json',
 '/content/drive/MyDrive/News topic classifier/saved_model/special_tokens_map.json',
 '/content/drive/MyDrive/News topic classifier/saved_model/vocab.txt',
 '/content/drive/MyDrive/News topic classifier/saved_model/added_tokens.json')

In [ ]:
from transformers import BertForSequenceClassification, BertTokenizer
import torch
model = BertForSequenceClassification.from_pretrained("/content/drive/MyDrive/News topic classifier/saved_model")
tokenizer = BertTokenizer.from_pretrained("/content/drive/MyDrive/News topic classifier/saved_model")
